# Generate and load synthetic mobility data

This notebook creates a deterministic, correlated dataset for all 13 tables and loads it through PostgreSQL `COPY`.

## Simulation model

Demand changes by hour, weekday, zone type, and simulation date. Driver selection favours nearby drivers while also considering quality and completed-trip experience. Offer acceptance responds to fare, pickup distance, driver behaviour, and time of day. Payment retries, cancellations, ratings, refunds, and reports are generated from the resulting ride outcome rather than sampled independently.

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path(r"C:\Python\projects\pgSQL")
load_dotenv(project_root / ".env", override=True)
sys.path.insert(0, str(project_root / "src"))

from MAna.database import connect_postgres, execute_query
from mansoura_mobility import (
    SimulationConfig,
    database_validation_queries,
    generate_synthetic_dataset,
    load_synthetic_dataset,
    validation_report,
)

db = connect_postgres(driver="psycopg2")

[OK] Connected to database: postgresql+psycopg2://localhost/mansoura_mobility


## Configuration

The fixed seed and end timestamp make the generated rows reproducible. The default scale produces about 1,000 passengers, 100 drivers, 14,000 offers, and roughly 10,000 accepted rides.

In [2]:
config = SimulationConfig(
    seed=20260830,
    passenger_count=1_000,
    driver_count=100,
    dual_role_count=20,
    offer_count=14_000,
    simulation_days=180,
    simulation_end="2026-08-30T23:59:00+03:00",
)
config

SimulationConfig(seed=20260830, passenger_count=1000, driver_count=100, dual_role_count=20, offer_count=14000, simulation_days=180, simulation_end='2026-08-30T23:59:00+03:00')

In [3]:
dataset = generate_synthetic_dataset(config)
dataset.counts()

,table_name,rows
0,accounts,1080
1,passengers,1000
2,drivers,100
3,zones,15
4,vehicles,127
5,zone_routes,210
6,offers,14000
7,rides,10617
8,payment_attempts,12276
9,refunds,275


## In-memory validation

Generation stops before loading if a relationship, amount, rating, or offer-to-ride rule fails.

In [4]:
validation_report(dataset)

,check,passed,detail
0,unique account contacts,True,"1,080 accounts"
1,roles reference accounts,True,passenger and driver shared keys
2,offer vehicle belongs to driver,True,100 used driver-vehicle pairs
3,routes reference zones,True,210 positive directional routes
4,accepted offers map one-to-one to rides,True,"10,617 rides from 14,000 offers"
5,open offers have no decision,True,pending and negotiating rows
6,payment amounts exceed 25 EGP,True,"12,276 payment attempts"
7,payment attempts are numbered per ride,True,unique attempt numbers and provider references
8,refunds do not exceed captured payments,True,275 refunds
9,rating scores stay in range,True,all supplied scores are 1–5


In [5]:
dataset.tables["offers"]["offer_status"].value_counts(dropna=False).to_frame("offers")

,offers
offer_status,
ACCEPTED,10617
DECLINED,1943
WITHDRAWN,824
AUTO_CANCELLED,616


## Load into PostgreSQL

The database is already populated with the default seed. Loading is disabled so a full notebook run remains safe. Enable `LOAD_TO_POSTGRESQL` for a new load, and enable replacement only for an intentional full synthetic-data rebuild.

In [6]:
LOAD_TO_POSTGRESQL = False
REPLACE_EXISTING_DATA = False

if LOAD_TO_POSTGRESQL:
    load_report = load_synthetic_dataset(
        dataset,
        db,
        replace_existing=REPLACE_EXISTING_DATA,
    )
else:
    load_report = dataset.counts().rename(columns={"rows": "generated_rows"})
load_report

,table_name,generated_rows
0,accounts,1080
1,passengers,1000
2,drivers,100
3,zones,15
4,vehicles,127
5,zone_routes,210
6,offers,14000
7,rides,10617
8,payment_attempts,12276
9,refunds,275


## PostgreSQL validation

These summaries are calculated in SQL after loading.

In [7]:
validation_sql = database_validation_queries()

In [8]:
execute_query(validation_sql["row_counts"], db, return_results=True)

,table_name,rows
0,accounts,1080
1,driver_passenger_ratings,5435
2,drivers,100
3,offers,14000
4,passenger_driver_ratings,6976
5,passengers,1000
6,payment_attempts,12276
7,refunds,275
8,reports,450
9,rides,10617


In [9]:
execute_query(validation_sql["offer_outcomes"], db, return_results=True)

,offer_status,offers,share_pct
0,ACCEPTED,10617,75.839996
1,DECLINED,1943,13.880000
2,WITHDRAWN,824,5.890000
3,AUTO_CANCELLED,616,4.400000


In [10]:
execute_query(validation_sql["ride_outcomes"], db, return_results=True)

,ride_status,rides,share_pct
0,COMPLETED,9728,91.629997
1,CANCELLED_BEFORE_START,572,5.390000
2,TERMINATED_EARLY,316,2.980000
3,AWAITING_PAYMENT,1,0.010000


In [11]:
execute_query(validation_sql["integrity"], db, return_results=True)

,rides_from_nonaccepted_offers,duplicate_offer_rides
0,0,0
